In [1]:
import pandas as pd
import numpy as np
import os
import sys; sys.path.append("/data/jerrylee/pjt/BIGFAM.v.0.1")
from BIGFAM import tools
from BIGFAM.obj1 import _slopeSig

In [2]:
sources = ["UKB", "GS"]

# Step 1. FRLog-reg

In [55]:
# filiter out phenotype which has too small n_pair
df_ps = pd.DataFrame(columns=["cohort", "pheno"])

# pheno list
for src in sources:
    frreg_path = f"/data/jerrylee/pjt/BIGFAM.v.0.1/data/{src}/frreg/DOR"
    p_fns = os.listdir(frreg_path)
    
    for p in p_fns:
        if p == "tmp":
            continue
        fn = frreg_path + "/" + p
        tmp = pd.read_csv(fn, sep='\t')
        
        if sum(tmp["n"] < 200): # less than 100 pairs
            continue
        
        df_ps.loc[len(df_ps)] = [src, p.split(".")[0]]

In [56]:
# load FRlogreg
df_frlog_mrg = pd.DataFrame()
for src in sources:
    frlog_path = f"/data/jerrylee/pjt/BIGFAM.v.0.1/data/{src}/FRLogreg"
    fns = os.listdir(frlog_path)
    fns = [fn for fn in fns if fn.split(".")[-1] == "FRLOG"]
    
    tmp_frlog = pd.DataFrame()
    for fn in fns:
        p = fn.split(".")[0]
        
        if p == "tmp":
            continue
        
        if p not in df_ps.loc[df_ps["cohort"] == src, "pheno"].to_list():
            continue
            
        # load FRLog-reg results
        frlog_fn = f"{frlog_path}/{fn}"
        df_slopetest = pd.read_csv(frlog_fn, sep='\t')
        df_slopetest_wide = tools.long2wide(df_slopetest)
        
        # annotation
        df_slopetest_wide["pheno"] = p
        df_slopetest_wide["sig"] = "None"
        if (df_slopetest_wide["lower_slope"].values[0] > 1):
            df_slopetest_wide["sig"] = "High"
        if (df_slopetest_wide["upper_slope"].values[0] < 1):
            df_slopetest_wide["sig"] = "Low"
        
        tmp_frlog = pd.concat([tmp_frlog, df_slopetest_wide], axis=0)
    
    tmp_frlog["cohort"] = src
    df_frlog_mrg = (pd.concat([df_frlog_mrg, tmp_frlog], axis=0)
                    .reset_index(drop=True))

df_frlog_mrg

,slope,lower_slope,upper_slope,intercept,lower_intercept,upper_intercept,pheno,sig,cohort
0,0.931290,0.890439,0.980765,-0.971320,-1.045296,-0.889537,Body_mass_index__BMI_,Low,UKB
1,1.023568,0.905393,1.115254,-1.754704,-1.973401,-1.636776,Creatinine,None,UKB
2,0.945760,0.890510,1.004999,-1.118566,-1.222833,-1.017913,Impedance_of_arm__right_,None,UKB
3,1.203189,1.124266,1.288758,-0.671934,-0.804818,-0.550196,Impedance_of_leg__left_,High,UKB
4,0.918686,0.873524,0.973581,-1.309409,-1.411195,-1.218170,IGF-1,Low,UKB
...,...,...,...,...,...,...,...,...,...
130,0.748723,0.424625,1.273799,-1.779015,-2.294087,-1.126548,FEF,None,GS
131,0.575063,0.336116,0.821931,-2.210741,-2.612877,-1.864333,Heart_Rate,Low,GS
132,0.664120,0.560002,0.761676,-0.597651,-0.738686,-0.442837,expected,Low,GS
133,0.416979,0.243413,0.677338,-2.381804,-2.613426,-2.035720,max_leg,Low,GS


In [57]:
df_frlog_mrg.to_csv(
    f"/data/jerrylee/pjt/BIGFAM.v.0.1/figures/data/frlogreg.v2.tsv",
    sep='\t',
    index=False)

## Step 1.1 check

In [58]:
df_frlog_mrg = pd.read_csv(
    f"/data/jerrylee/pjt/BIGFAM.v.0.1/figures/data/frlogreg.v2.tsv",
    sep='\t',
)
df_frlog_mrg.shape

(135, 9)

In [67]:
df_frlog_mrg.groupby(["sig"]).size()

sig
High    18
Low     54
None    63
dtype: int64

In [59]:
df_frlog_mrg.groupby(["cohort", "sig"]).size()

cohort  sig 
GS      Low     15
        None    11
UKB     High    18
        Low     39
        None    52
dtype: int64

# Step 2. Prediction (obj1)

In [46]:
# functions to load data
def load_bigfam(path):
    pheno_fns = os.listdir(path)
    phenos = [pheno_fn.split(".")[0] for pheno_fn in pheno_fns]
    phenos = set(phenos)
    
    df_bigfam = pd.DataFrame()
    for pheno in phenos:
        try:
            tmp = pd.read_csv(f"{path}/{pheno}.GSW", sep='\t')
            tmp["param"] = ["G_BIGFAM", "S_BIGFAM", "w_BIGFAM", "mse_train", "mse_test"]
            tmp_wide = tools.long2wide(tmp)
            
            tmp_sig = pd.read_csv(f"{path}/{pheno}.FRLOG_raw", sep='\t')
            sig = _slopeSig(tmp_sig["slope"])
            tmp_wide["pheno"] = pheno
            tmp_wide["sig"] = sig
            
            df_bigfam = pd.concat([df_bigfam, tmp_wide], axis=0)
        except:
            continue
    
    return df_bigfam.reset_index(drop=True)

In [49]:
# load FRlogreg
df_pred = pd.DataFrame()

for src in sources:
    pred_path = f"/data/jerrylee/pjt/BIGFAM.v.0.1/data/{src}/obj1"
    frlog_path = f"/data/jerrylee/pjt/BIGFAM.v.0.1/data/{src}/FRLogreg"
    fns = os.listdir(pred_path)
    fns = [fn for fn in fns if fn.split(".")[-1] == "GSW"]
    
    tmp_pred = pd.DataFrame()
    for fn in fns:
        p = fn.split(".")[0]
        
        if p == "tmp":
            continue
        
        if p not in df_ps.loc[df_ps["cohort"] == src, "pheno"].to_list():
            continue
            
        # load FRLog-reg results
        pred_fn = f"{pred_path}/{fn}"
        tmp = pd.read_csv(pred_fn, sep='\t')
        tmp["param"] = ["G_BIGFAM", "S_BIGFAM", "w_BIGFAM", "mse_train", "mse_test"]
        tmp_wide = tools.long2wide(tmp)
        tmp_sig = pd.read_csv(f"{frlog_path}/{p}.FRLOG_raw", sep='\t')
        sig = _slopeSig(tmp_sig["slope"])
        tmp_wide["pheno"] = p
        tmp_wide["sig"] = sig
        
        tmp_pred = pd.concat([tmp_pred, tmp_wide], axis=0)
    
    tmp_pred["cohort"] = src
    df_pred = (pd.concat([df_pred, tmp_pred], axis=0)
               .reset_index(drop=True))

df_pred

,G_BIGFAM,lower_G_BIGFAM,upper_G_BIGFAM,S_BIGFAM,lower_S_BIGFAM,upper_S_BIGFAM,w_BIGFAM,lower_w_BIGFAM,upper_w_BIGFAM,mse_train,lower_mse_train,upper_mse_train,mse_test,lower_mse_test,upper_mse_test,pheno,sig,cohort
0,0.625072,0.623822,0.626010,0.012932,0.012664,0.013163,0.95,0.95,0.95000,1.430399,1.360756,1.483043,0.156647,0.104312,0.226269,Whole_body_water_mass,Low,UKB
1,0.157405,0.156654,0.158337,0.017552,0.017325,0.017741,0.95,0.95,0.95000,20.133582,19.069972,20.791932,2.188345,1.532388,3.257606,Monocyte_count,Low,UKB
2,0.262343,0.205391,0.284595,0.013680,0.000001,0.041641,0.46,0.40,0.58000,2.601725,2.431669,2.749792,0.278115,0.171268,0.416885,Creatinine,None,UKB
3,0.202208,0.046325,0.208625,0.041501,0.040553,0.121528,0.48,0.43,0.48525,4.130742,3.922101,4.482038,0.457043,0.243451,0.669903,Calcium,None,UKB
4,0.458526,0.457183,0.459740,0.037280,0.037027,0.037622,0.95,0.95,0.95000,2.561443,2.426515,2.650125,0.279119,0.190487,0.414052,"Forced_vital_capacity__FVC_,_Best_measure",Low,UKB
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
130,0.455049,0.451846,0.457618,0.049490,0.048517,0.050764,0.95,0.95,0.95000,8.589694,8.281987,8.853984,0.957657,0.692795,1.265257,HDL_cholesterol,Low,GS
131,0.096884,0.095660,0.098432,0.040679,0.039775,0.041338,0.95,0.95,0.95000,269.010931,228.354445,276.289480,25.664916,18.450950,66.413729,avg_sys,Low,GS
132,0.175468,0.173421,0.177674,0.030695,0.029801,0.031550,0.95,0.95,0.95000,92.888363,89.296112,95.727684,10.173596,7.343407,13.787060,Total_cholesterol,Low,GS
133,0.066761,0.000001,0.084605,0.137953,0.129338,0.187261,0.60,0.53,0.60000,22.546879,20.324521,25.426939,2.259238,1.167123,4.531426,FEF,None,GS


In [50]:
df_pred.to_csv(
    f"/data/jerrylee/pjt/BIGFAM.v.0.1/figures/data/obj1.v2.tsv",
    sep='\t',
    index=False)

## Step 2.1 check

In [68]:
df_pred = pd.read_csv(
    "/data/jerrylee/pjt/BIGFAM.v.0.1/figures/data/obj1.v2.tsv",
    sep='\t',
)
df_pred.shape

(135, 18)

In [102]:
# tt = df_pred[(df_pred["lower_G_BIGFAM"] > 1e-6) & (df_pred["lower_S_BIGFAM"] > 1e-6)].copy()
tt = df_pred[(df_pred["lower_G_BIGFAM"] > 1e-6)].copy()
tt["ww_BIGFAM"] = 1 / tt["w_BIGFAM"]
tt.shape

(103, 19)

In [97]:
tt[["G_BIGFAM", "S_BIGFAM", "ww_BIGFAM"]].agg(["mean", "std", "size"])

,G_BIGFAM,S_BIGFAM,ww_BIGFAM
mean,0.327948,0.040880,10.849764
std,0.170700,0.042918,28.580169
size,98.000000,98.000000,98.000000


In [98]:
tt.groupby("sig")[["G_BIGFAM", "S_BIGFAM", "ww_BIGFAM"]].agg(["mean", "std", "size"])

G_BIGFAM                 S_BIGFAM                 ww_BIGFAM             \
          mean       std size      mean       std size       mean        std   
sig                                                                            
High  0.328485  0.151645   13  0.057043  0.060402   13  71.888293  44.127473   
Low   0.352367  0.192399   48  0.042528  0.046159   48   1.129431   0.217951   
None  0.296080  0.143804   37  0.033064  0.028365   37   2.013956   0.374268   

           
     size  
sig        
High   13  
Low    48  
None   37

In [99]:
tt.groupby(["cohort", "sig"])[["G_BIGFAM", "S_BIGFAM", "ww_BIGFAM"]].agg(["mean", "std", "size"])

G_BIGFAM                 S_BIGFAM                 ww_BIGFAM  \
                 mean       std size      mean       std size       mean   
cohort sig                                                                 
GS     Low   0.217755  0.173327   12  0.060139  0.022297   12   1.052632   
       None  0.156245       NaN    1  0.030758       NaN    1   1.818182   
UKB    High  0.328485  0.151645   13  0.057043  0.060402   13  71.888293   
       Low   0.397237  0.178840   36  0.036657  0.050627   36   1.155031   
       None  0.299965  0.143861   36  0.033128  0.028765   36   2.019394   

                             
                   std size  
cohort sig                   
GS     Low    0.000000   12  
       None        NaN    1  
UKB    High  44.127473   13  
       Low    0.247170   36  
       None   0.378091   36

In [100]:
tt[tt["sig"] == "High"].sort_values(by=["w_BIGFAM", "upper_w_BIGFAM"])[["pheno", "w_BIGFAM", "upper_w_BIGFAM"]]

,pheno,w_BIGFAM,upper_w_BIGFAM
5,SHBG,0.010,0.01000
29,Hip_circumference,0.010,0.01000
31,Impedance_of_whole_body,0.010,0.01000
41,Arm_fat_mass__right_,0.010,0.01000
64,Aspartate_aminotransferase,0.010,0.01000
108,Arm_fat_mass__left_,0.010,0.01000
23,Glycated_haemoglobin__HbA1c_,0.010,0.02525
66,Total_bilirubin,0.010,0.02525
33,Red_blood_cell__erythrocyte__distribution_width,0.010,0.12525
67,Alanine_aminotransferase,0.045,0.13000


In [101]:
tt.sort_values(by=["S_BIGFAM"])

,G_BIGFAM,lower_G_BIGFAM,upper_G_BIGFAM,S_BIGFAM,lower_S_BIGFAM,upper_S_BIGFAM,w_BIGFAM,lower_w_BIGFAM,upper_w_BIGFAM,mse_train,lower_mse_train,upper_mse_train,mse_test,lower_mse_test,upper_mse_test,pheno,sig,cohort,ww_BIGFAM
92,0.322178,0.225172,0.324078,0.005307,0.003914,0.050074,0.47,0.400,0.600,7.051285,6.760856,7.369593,0.773067,0.497061,1.047143,Apolipoprotein_B,None,UKB,2.127660
85,0.033828,0.033167,0.038136,0.006763,0.006603,0.007258,0.60,0.450,0.600,326.790541,289.187266,346.542433,32.378976,13.357896,70.323395,Interpolated_Age_of_participant_when_non-cance...,None,UKB,1.666667
12,0.313111,0.224734,0.327463,0.006810,0.001444,0.052890,0.60,0.419,0.600,4.798834,4.547196,5.066990,0.522822,0.335620,0.822755,Neutrophill_count,None,UKB,1.666667
71,0.034139,0.031312,0.040188,0.006824,0.002858,0.006941,0.40,0.400,0.571,487.988047,456.549439,512.333813,53.425814,29.145449,85.730406,Basophill_percentage,None,UKB,2.500000
61,0.311496,0.223858,0.322385,0.007039,0.003850,0.048520,0.60,0.400,0.600,9.193387,8.900769,9.514308,1.011989,0.714981,1.353325,Haemoglobin_concentration,None,UKB,1.666667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
69,0.221610,0.166686,0.442023,0.119918,0.011293,0.146952,0.56,0.550,0.950,0.661117,0.614295,0.706679,0.069872,0.037825,0.123644,Mean_corpuscular_haemoglobin,Low,UKB,1.785714
80,0.018829,0.008358,0.310659,0.156313,0.012393,0.161359,0.55,0.550,0.950,1.209083,1.148205,1.262819,0.131056,0.069741,0.206945,Hand_grip_strength__left_,Low,UKB,1.818182
7,0.124070,0.112879,0.333197,0.174257,0.066659,0.179778,0.48,0.470,0.480,6.018401,5.841076,6.232745,0.665333,0.528138,0.828368,Alkaline_phosphatase,None,UKB,2.083333
51,0.567949,0.257959,0.733260,0.244915,0.164181,0.398178,0.70,0.630,0.780,0.077998,0.071336,0.090949,0.007379,0.004447,0.014662,Standing_height,Low,UKB,1.428571


In [104]:
tt[tt["pheno"] == "Impedance_of_leg__right_"][["pheno", "G_BIGFAM", "S_BIGFAM", "ww_BIGFAM"]]

,pheno,G_BIGFAM,S_BIGFAM,ww_BIGFAM
103,Impedance_of_leg__right_,0.050981,0.255088,2.325581


In [105]:
tt[tt["pheno"] == "Standing_height"][["pheno", "G_BIGFAM", "S_BIGFAM", "ww_BIGFAM"]]

,pheno,G_BIGFAM,S_BIGFAM,ww_BIGFAM
51,Standing_height,0.567949,0.244915,1.428571
